# ARIA - LITE

ARIA Lite is a lightweight GraphRAG-based biomedical research assistant focused on breast cancer AI literature.

The project combines two retrieval paradigms:

1. Semantic Retrieval
   Dense vector embeddings are used to retrieve papers semantically related to a user query.

2. Graph-Based Retrieval
   Biomedical entities such as genes and drugs are extracted from papers and represented as relationships in a lightweight knowledge graph.

By combining these two approaches, the system aims to provide more grounded and explainable retrieval compared to traditional vector-only RAG systems.

The project is intentionally scoped for rapid iteration and learning:
- ~300-500 PubMed papers
- Abstract-only corpus
- Lightweight graph construction
- Citation-grounded responses

Core technologies:
- PubMed / Entrez API
- SciSpacy
- Sentence Transformers
- FAISS
- Python + Google Colab

End Goal:
Build a small but functional biomedical GraphRAG system capable of retrieving relevant breast cancer AI papers and generating grounded answers with PMID citations.

# 7_hybrid_retrieval.ipynb

Goal
----
This notebook combines:

1. Semantic Retrieval
   (FAISS vector similarity)

2. Graph Retrieval
   (Knowledge graph traversal)

to build a Hybrid GraphRAG Retrieval System.

Why Hybrid Retrieval?
---------------------
Vector retrieval is good at:
- semantic similarity
- contextual meaning

But it can miss:
- biological relationships
- graph structure
- concept connectivity

Graph retrieval is good at:
- entity relationships
- biomedical neighborhood expansion
- structural reasoning

But it can retrieve:
- broad/noisy papers
- graph hubs

Combining BOTH gives:
- semantic precision
- structural awareness

Pipeline
--------
User Query
    ↓
Embedding Search (FAISS)
    ↓
Graph Retrieval
    ↓
Merge + Rank
    ↓
Final Retrieved Papers

Output
------
A ranked set of biomedical papers
using both:
- vector similarity
- graph connectivity

This notebook is the core retrieval engine
of ARIA-Lite GraphRAG.

In [1]:
# ============================================================
# SECTION 1 — Install Dependencies
# ============================================================

!pip install sentence_transformers faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 39.7 MB/s eta 0:00:00


In [2]:
# ============================================================
# SECTION 2 — Import
# ============================================================

from google.colab import drive
import os
import json
import pickle
import faiss
import networkx as nx
import numpy as np

from collections import Counter
from sentence_transformers import SentenceTransformer

In [3]:
# ============================================================
# SECTION 3 — Paths
# ============================================================

drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite"

# --------------------------------------------------
# Papers
# --------------------------------------------------

PAPERS_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "clean_papers.json"
)

# --------------------------------------------------
# Graph
# --------------------------------------------------

GRAPH_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "aria_lite_graph_v2.pkl"
)

# --------------------------------------------------
# Vector Index Files
# --------------------------------------------------

INDICES_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "indices"
)

FAISS_INDEX_PATH = os.path.join(
    INDICES_DIR,
    "faiss_index.bin"
)

PMIDS_PATH = os.path.join(
    INDICES_DIR,
    "pmids.pkl"
)

TEXTS_PATH = os.path.join(
    INDICES_DIR,
    "texts.pkl"
)

Mounted at /content/drive


In [4]:
import json
import pickle
import faiss

# --------------------------------------------------
# Load papers
# --------------------------------------------------

with open(PAPERS_PATH, "r") as f:
    papers = json.load(f)

print("Loaded papers:", len(papers))

# --------------------------------------------------
# Load graph
# --------------------------------------------------

with open(GRAPH_PATH, "rb") as f:
    G = pickle.load(f)

print("\nGraph loaded")

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# --------------------------------------------------
# Load FAISS index
# --------------------------------------------------

index = faiss.read_index(FAISS_INDEX_PATH)

print("\nFAISS index loaded")

# --------------------------------------------------
# Load PMID mapping
# --------------------------------------------------

with open(PMIDS_PATH, "rb") as f:
    pmids = pickle.load(f)

print("PMIDs loaded:", len(pmids))

# --------------------------------------------------
# Load stored texts
# --------------------------------------------------

with open(TEXTS_PATH, "rb") as f:
    texts = pickle.load(f)

print("Texts loaded:", len(texts))

Loaded papers: 475

Graph loaded
Nodes: 1108
Edges: 19839

FAISS index loaded
PMIDs loaded: 475
Texts loaded: 475


In [5]:
# ============================================================
# SECTION 5 — Load LLM model
# ============================================================

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


In [6]:
# ============================================================
# SECTION 6 — Paper Lookup Dictionary
# ============================================================

paper_lookup = {}

for p in papers:
    paper_lookup[p["pmid"]] = p

print("Paper lookup built")

Paper lookup built


In [7]:
# ============================================================
# SECTION 7 — Simple Entity Extractor
# ============================================================

IGNORE_ENTITIES = {
}


def simple_entity_extract(text):

    text = text.lower()

    found = []

    for node, data in G.nodes(data=True):

        if data.get("type") != "entity":
            continue

        if node in IGNORE_ENTITIES:
            continue

        if node in text:
            found.append(node)

    return list(set(found))

In [8]:
# ============================================================
# SECTION 8 — Vector Retrieval
# ============================================================

def vector_retrieval(query, top_k=10):

    query_embedding = embedding_model.encode([query])

    D, I = index.search(
        np.array(query_embedding).astype("float32"),
        top_k
    )

    results = []

    for idx, score in zip(I[0], D[0]):

        pmid = pmids[idx]

        results.append({
            "pmid": pmid,
            "score": float(score)
        })

    return results

In [9]:
# ============================================================
# SECTION 9 — Graph Expansion
# ============================================================

def expand_from_entity(entity):

    papers = set()
    related_entities = set()

    if entity not in G:
        return {
            "direct_papers": papers,
            "related_entities": related_entities
        }

    neighbors = list(G.neighbors(entity))

    for n in neighbors:

        node_data = G.nodes[n]

        if node_data["type"] == "paper":
            papers.add(n)

            second_neighbors = list(G.neighbors(n))

            for s in second_neighbors:

                if s == entity:
                    continue

                if G.nodes[s]["type"] == "entity":
                    related_entities.add(s)

    return {
        "direct_papers": papers,
        "related_entities": related_entities
    }

In [10]:
# ============================================================
# SECTION 10 — Graph Retrieval
# ============================================================

def graph_retrieval(query):

    entities = simple_entity_extract(query)

    paper_scores = Counter()
    all_entities = set()

    for ent in entities:

        if ent not in G:
            continue

        expansion = expand_from_entity(ent)

        all_entities.update(
            expansion["related_entities"]
        )

        degree = G.degree(ent)

        score = 1 / (degree + 1)

        for paper_id in expansion["direct_papers"]:

            paper_scores[paper_id] += score

    ranked_papers = sorted(
        paper_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return {
        "entities_found": entities,
        "papers": ranked_papers,
        "related_entities": list(all_entities)
    }

In [11]:
# ============================================================
# SECTION 11 — Hybrid Retrieval
# ============================================================

def hybrid_retrieval(
    query,
    vector_weight=0.7,
    graph_weight=0.3,
    top_k=10
):

    # ----------------------------------------
    # Vector Retrieval
    # ----------------------------------------

    vector_results = vector_retrieval(
        query,
        top_k=top_k
    )

    # normalize vector scores
    vector_scores = {}

    max_vector = max(
        [x["score"] for x in vector_results]
    )

    for r in vector_results:

        norm_score = r["score"] / max_vector

        vector_scores[r["pmid"]] = norm_score

    # ----------------------------------------
    # Graph Retrieval
    # ----------------------------------------

    graph_results = graph_retrieval(query)

    graph_scores = {}

    if len(graph_results["papers"]) > 0:

        max_graph = max(
            [x[1] for x in graph_results["papers"]]
        )

        for pmid, score in graph_results["papers"]:

            norm_score = score / max_graph

            graph_scores[pmid] = norm_score

    # ----------------------------------------
    # Merge Scores
    # ----------------------------------------

    final_scores = {}

    all_pmids = set(
        list(vector_scores.keys()) +
        list(graph_scores.keys())
    )

    for pmid in all_pmids:

        v_score = vector_scores.get(pmid, 0)
        g_score = graph_scores.get(pmid, 0)

        final_score = (
            vector_weight * v_score +
            graph_weight * g_score
        )

        final_scores[pmid] = final_score

    # ----------------------------------------
    # Rank
    # ----------------------------------------

    ranked = sorted(
        final_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return {
        "query": query,
        "entities_found": graph_results["entities_found"],
        "related_entities": graph_results["related_entities"],
        "papers": ranked[:top_k]
    }

In [12]:
# ============================================================
# SECTION 12 — Display Results
# ============================================================

def show_results(results):

    print("\n=== QUERY ===")
    print(results["query"])

    print("\n=== ENTITIES ===")
    print(results["entities_found"])

    print("\n=== RELATED ENTITIES ===")
    print(results["related_entities"][:15])

    print("\n=== TOP PAPERS ===")

    for pmid, score in results["papers"]:

        if pmid not in paper_lookup:
            continue

        p = paper_lookup[pmid]

        print("\n" + "=" * 80)

        print("PMID:", pmid)
        print("Hybrid Score:", round(score, 4))

        print("\nTitle:")
        print(p["title"])

In [13]:
# ============================================================
# SECTION 13 — Test Hybrid Retrieval
# ============================================================

query = "HER2 immunotherapy in breast cancer"

results = hybrid_retrieval(
    query,
    vector_weight=0.7,
    graph_weight=0.3,
    top_k=5
)

show_results(results)


=== QUERY ===
HER2 immunotherapy in breast cancer

=== ENTITIES ===
['er', 'breast', 'cancer', 'breast cancer', 'her2', 'st', 'th']

=== RELATED ENTITIES ===
['colorectal cancers', 'random forest', 'akt1', 'logistic regression', 'tc', 'itgb4', 'triple-negative', 'molecular docking', 'esr1', 'bcl-2', 'tumour-to-tumour', 'mastl', 'xbp1', 'auroc', 'til']

=== TOP PAPERS ===

PMID: 42018935
Hybrid Score: 0.7265

Title:
Focusing on the Advances and Challenges of HER2-Targeted Therapy: Cutting-Edge Breakthroughs in Neoadjuvant Treatment for HER2-Positive Breast Cancer.

PMID: 41985721
Hybrid Score: 0.7063

Title:
Pathological response to herceptin-containing neoadjuvant therapy in HER2 IHC2+/ISH+ and IHC3+ early-stage invasive ductal carcinoma.

PMID: 42025215
Hybrid Score: 0.702

Title:
Toward Sustainable Care in HER2-Positive Metastatic Breast Cancer: Emerging Evidence for Treatment De-escalation.

PMID: 42016158
Hybrid Score: 0.6495

Title:
Inflammatory and Nutritional Biomarkers Predict